# Quickstart: Querying PDF With Astra and LangChain

### A question-answering demo using Astra DB and LangChain, powered by Vector Search

Import the packages you'll need:

In [2]:
# ============================================================
# STEP 3 - IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from pypdf import PdfReader

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
    ChatHuggingFace,
)

from langchain_astradb import AstraDBVectorStore

from langchain_text_splitters import CharacterTextSplitter

c:\Data Science 2026\Project Repository\AI_Agents_LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# STEP 4 - ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")

ASTRA_DB_APPLICATION_TOKEN = os.getenv(
    "ASTRA_DB_APPLICATION_TOKEN"
)

ASTRA_DB_API_ENDPOINT = os.getenv(
    "ASTRA_DB_API_ENDPOINT"
)

print("HF token loaded:", bool(HF_TOKEN))
print("Astra token loaded:", bool(ASTRA_DB_APPLICATION_TOKEN))
print("Astra endpoint loaded:", bool(ASTRA_DB_API_ENDPOINT))

HF token loaded: True
Astra token loaded: True
Astra endpoint loaded: True


In [4]:


# ============================================================
# STEP 5 - READ PDF
# ============================================================

pdfreader = PdfReader("C:\\Data Science 2026\\Project Repository\\AI_Agents_LangGraph\\10.PDFQuery_LangChain\\Budget_Speech.pdf")

raw_text = ""

for page in pdfreader.pages:

    content = page.extract_text()

    if content:
        raw_text += content + "\n"

print(f"Extracted {len(raw_text)} characters")

Extracted 92141 characters


In [5]:
print(raw_text[:3000])

GOVERNMENT OF INDIA
BUDGET 2025-2026
SPEECH
OF
NIRMALA SITHARAMAN
MINISTER OF FINANCE
February 1,  2025
 
CONTENTS 
 
PART – A 
 Page No. 
Introduction 1 
Budget Theme 1 
Agriculture as the 1st engine 3 
MSMEs as the 2nd engine 6 
Investment as the 3rd engine 8 
A. Investing in People 8 
B. Investing in the Economy 10 
C. Investing in Innovation 14 
Exports as the 4th engine 15 
Reforms as the Fuel 16 
Fiscal Policy 18 
 
 
PART – B 
Indirect taxes 20 
Direct Taxes  23 
 
Annexure to Part-A 29 
Annexure to Part-B 31 
 
  
 
 
Budget 2025-2026 
 
Speech of 
Nirmala Sitharaman 
Minister of Finance 
February 1, 2025 
Hon’ble Speaker,  
 I present the Budget for 2025-26. 
Introduction 
1. This Budget continues our Government’s efforts to: 
a) accelerate growth,  
b) secure inclusive development,  
c) invigorate private sector investments,  
d) uplift household sentiments, and 
e) enhance spending power of India’s rising middle class.  
2. Together, we embark on a journey to unlock our nati

In [6]:
# ============================================================
# STEP 6 - TEXT SPLITTING
# ============================================================

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)

texts = text_splitter.split_text(raw_text)

print(f"Created {len(texts)} chunks")

Created 153 chunks


In [7]:
# ============================================================
# STEP 7 - HUGGING FACE EMBEDDINGS
# ============================================================

embedding_model = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model,
)

print("Hugging Face embedding model configured")

Hugging Face embedding model configured


In [8]:
# ============================================================
# STEP 8 - TEST CLOUD EMBEDDINGS
# ============================================================

test_embedding = embeddings.embed_query(
    "What is India's GDP?"
)

print("Embedding dimensions:", len(test_embedding))
print(test_embedding[:10])

Embedding dimensions: 384
[0.011114082299172878, 0.02135222591459751, -0.09249677509069443, 0.06366261094808578, -0.029749928042292595, -0.04083123058080673, 0.06675340235233307, 0.013968830928206444, 0.0006548038218170404, -0.010340639390051365]


In [9]:
# ============================================================
# STEP 9 - ASTRA DB VECTOR STORE
# ============================================================

vector_store = AstraDBVectorStore(
    embedding=embeddings,
    collection_name="budget_speech_demo",
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

print("Astra DB vector store created")

Astra DB vector store created


In [10]:
# ============================================================
# STEP 10 - INSERT PDF CHUNKS
# ============================================================

vector_store.add_texts(texts)

print(f"Inserted {len(texts)} chunks into Astra DB")

Inserted 153 chunks into Astra DB


In [11]:
# ============================================================
# STEP 11 - RETRIEVER
# ============================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever created")

Retriever created


In [12]:
# ============================================================
# STEP 12 - TEST RETRIEVAL
# ============================================================

query = "What is the current GDP?"

docs = retriever.invoke(query)

print(f"Retrieved {len(docs)} documents")

Retrieved 4 documents


In [13]:
for i, doc in enumerate(docs, start=1):

    print(f"\n========== DOCUMENT {i} ==========")

    print(doc.page_content[:1000])


========== DOCUMENT 1 ==========
Revised Estimates 2024-25 
110. The Revised Estimate of the total receipts other than borrowings is  
` 31.47 lakh crore, of which the net tax receipts are ` 25.57 lakh crore. The 
Revised Estimate of the total expenditure is ` 47.16 lakh crore, of which the 
capital expenditure is about ` 10.18 lakh crore. 
111. The Revised Estimate of the fiscal deficit is 4.8 per cent of GDP. 
 19  
 
Budget Estimates 2025-26 
112. Coming to 2025 -26, the total receipts other than  borrowings and the 
total expenditure are estimated at ` 34.96 lakh crore and ` 50.65 lakh crore 
respectively. The net tax receipts are estimated at ` 28.37 lakh crore. 
113. The fiscal deficit is estimated to be 4.4 per cent of GDP. 
114. To finance the fiscal deficit, the net market borrowings from dated

========== DOCUMENT 2 ==========
Revised Estimates 2024-25 
110. The Revised Estimate of the total receipts other than borrowings is  
` 31.47 lakh crore, of which the net tax receipt

Create the LangChain embedding and LLM objects for later usage:

In [14]:
# ============================================================
# STEP 13 - HUGGING FACE CLOUD LLM
# ============================================================

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    provider="auto",
    max_new_tokens=512,
    temperature=0.1,
    huggingfacehub_api_token=HF_TOKEN,
)

print("Hugging Face cloud LLM configured")

Hugging Face cloud LLM configured


Create your LangChain vector store ... backed by Astra DB!

In [15]:
# ============================================================
# STEP 14 - CHAT MODEL
# ============================================================

chat_model = ChatHuggingFace(
    llm=llm
)

print("Hugging Face chat model ready")

Hugging Face chat model ready


In [16]:
# ============================================================
# STEP 15 - TEST LLM
# ============================================================

response = chat_model.invoke(
    "What is artificial intelligence?"
)

print(response.content)

**Artificial intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and even creativity. In essence, AI tries to give machines the ability to “think” and act autonomously or semi‑autonomously.

---

## Core Concepts

| Concept | What It Means | Typical Techniques |
|---------|---------------|--------------------|
| **Machine Learning (ML)** | Algorithms that improve performance on a task through experience (data). | Supervised, unsupervised, reinforcement learning; decision trees, SVMs, k‑NN, etc. |
| **Deep Learning** | A subset of ML using multi‑layer neural networks to automatically learn hierarchical features. | Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), Transformers. |
| **Natural Language Processing (NLP)** | Understanding and generating human language. | Token

In [19]:
# ============================================================
# STEP 16 - RAG PROMPT
# ============================================================

query_text = "Who is the Telugu poet name mentioned the the PDF?"

docs = retriever.invoke(query_text)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = f"""
Answer the question using only the context provided below.

If the answer is not available in the context,
say that you could not find the answer in the PDF.

Context:
{context}

Question:
{query_text}

Answer:
"""

print(prompt)


Answer the question using only the context provided below.

If the answer is not available in the context,
say that you could not find the answer in the PDF.

Context:
geopolitical headwinds suggest lower  global economic growth over the 
medium term. However, our aspiration for a Viksit Bharat inspires us, and the 
transformative work we have done during our Government’s first two terms 
guides us, to march forward resolutely.  
Budget Theme 
4. Our economy is the fastest-growing among all major global economies. 
Our development track record of the past 10 years and structural reforms have 
drawn global attention. Confidence in India’s capability and potential has only 
 2  
 
grown in this period. We see the next five years as a unique opportunity to 
realize ‘Sabka Vikas’, stimulating balanced growth of all regions. 
5. The great Telugu poet and playwright Gurajada Appa Rao had said,

geopolitical headwinds suggest lower  global economic growth over the 
medium term. However, our 

In [20]:
# ============================================================
# STEP 17 - GENERATE RAG ANSWER
# ============================================================

response = chat_model.invoke(prompt)

print(response.content)

The Telugu poet mentioned is **Gurajada Appa Rao**.


In [21]:
# ============================================================
# STEP 18 - INTERACTIVE QA LOOP
# ============================================================

first_question = True

while True:

    if first_question:

        query_text = input(
            "\nEnter your question (or type 'quit' to exit): "
        ).strip()

    else:

        query_text = input(
            "\nWhat's your next question (or type 'quit' to exit): "
        ).strip()


    # --------------------------------------------------------
    # Exit
    # --------------------------------------------------------

    if query_text.lower() == "quit":
        break


    # --------------------------------------------------------
    # Ignore empty questions
    # --------------------------------------------------------

    if query_text == "":
        continue


    first_question = False


    print(f'\nQUESTION: "{query_text}"')


    # ========================================================
    # 1. RETRIEVAL
    # ========================================================

    docs = retriever.invoke(query_text)


    # ========================================================
    # 2. CREATE CONTEXT
    # ========================================================

    context = "\n\n".join(
        doc.page_content
        for doc in docs
    )


    # ========================================================
    # 3. CREATE PROMPT
    # ========================================================

    prompt = f"""
Answer the question using only the context provided below.

If the answer is not available in the context,
say that you could not find the answer in the PDF.

Context:
{context}

Question:
{query_text}

Answer:
"""


    # ========================================================
    # 4. CALL HUGGING FACE CLOUD LLM
    # ========================================================

    response = chat_model.invoke(prompt)


    # ========================================================
    # 5. DISPLAY ANSWER
    # ========================================================

    print("\nANSWER:")
    print(response.content)


    # ========================================================
    # 6. DISPLAY RETRIEVED DOCUMENTS
    # ========================================================

    print("\nRELEVANT DOCUMENTS:")

    for i, doc in enumerate(docs, start=1):

        print(f"\n--- Document {i} ---")

        print(doc.page_content[:500])


QUESTION: "who is the telegu poet name"

ANSWER:
The Telugu poet mentioned is **Gurajada Appa Rao**.

RELEVANT DOCUMENTS:

--- Document 1 ---
geopolitical headwinds suggest lower  global economic growth over the 
medium term. However, our aspiration for a Viksit Bharat inspires us, and the 
transformative work we have done during our Government’s first two terms 
guides us, to march forward resolutely.  
Budget Theme 
4. Our economy is the fastest-growing among all major global economies. 
Our development track record of the past 10 years and structural reforms have 
drawn global attention. Confidence in India’s capability and potenti

--- Document 2 ---
geopolitical headwinds suggest lower  global economic growth over the 
medium term. However, our aspiration for a Viksit Bharat inspires us, and the 
transformative work we have done during our Government’s first two terms 
guides us, to march forward resolutely.  
Budget Theme 
4. Our economy is the fastest-growing among all major gl